In [1]:
import scanpy as sc
import hdf5plugin
import anndata as ad

import numpy as np
from numpy.random import choice
import pandas as pd

import matplotlib.pyplot as plt
import seaborn.objects  as so

from scipy.stats import chi2_contingency, pearsonr
import decoupler as dc

from tqdm import tqdm

/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [163]:
import importlib
importlib.reload(dc)

<module 'decoupler' from '/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/decoupler/__init__.py'>

# Load Data:

In [2]:
def ctrl_pert_split_dataset(norm_data: ad.AnnData, bad_perts : list[str] = None) -> tuple[list[str], dict[str,list[str]]]:
    """
    Split the norm_data.obs dataframe into control cells and each perturbation.

    Parameters
    ----------
    norm_data : anndata
        The anndata object containing the data.
    bad_perts : list
        A list of perturbations at least one of the models cannot process.

    Returns
    -------
    ctrl_i_vec : list[str]
        A vector of all indices of control cells in norm_data.obs (corresponds to the rows in norm_data.X).
    pert_i_dict : dict[str, list[str]]
        A dictionary with the perturbation name and the indices in norm_data.obs where this perturbation is found as key value pairs.
    """
    # Split df into perturbed and control cells
    
    by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]
    # Find indices for each perturbation
    pert_i_dict = {pert.iloc[0]["perturbation"]:pert.index.to_list() for pert in by_pert}
    # Extract the control cells separately
    ctrl_i_vec = pert_i_dict.pop("ctrl")

    # Optional: remove all perturbations that do not appear in a dataset
    if bad_perts is not None:
        for bp in bad_perts:
            pert_i_dict.pop(bp[:-5]) # the last 5 letters are +ctrl and need to be ignored
    
    return (ctrl_i_vec, pert_i_dict)

In [3]:
names = ["Norman19"]#,"Replogle22"]
models = ["pGRiNS", "Random"]
tests = ["Common DEGs", "Common highly expressed genes"]

In [4]:
"""
pgrins_full = sc.read_h5ad("Data/Projects/KeggoRo/perturb_norm_pert_reduced.h5ad")
pgrins = {}
pgrins["Norman19"] = pgrins_full[pgrins_full.obs["PertNum"]<=75]
pgrins["Replogle22"] = pgrins_full[pgrins_full.obs["PertNum"]==-1 | pgrins_full.obs["PertNum"]>75]
"""
exp = {}
exp["Norman19"] = sc.read_h5ad("Data/Experimental/Norman19/perturb_norm_subset_KeggoRo.h5ad")
exp["Replogle22"] = sc.read_h5ad("Data/Experimental/Replogle22/perturb_norm_subset_KeggoRo.h5ad")

perts = {}
pert_indices = {}
pert_means = {}
for name in names:
    perts[name] = list(exp[name].obs["perturbation"].unique())
    perts[name].remove("ctrl")
    pert_indices[name] = ctrl_pert_split_dataset(exp[name])
    pert_means[name] = {pert : np.asarray(np.mean(exp[name][pert_indices[name][1][pert]].layers["log1p"],axis=0)).squeeze() for pert in perts[name]}
    sc.tl.rank_genes_groups(exp[name],groupby="perturbation",reference="ctrl",rankby_abs=True,layer="log1p")
    #sc.tl.rank_genes_groups(pgrins[name],groupby="perturbation",reference="ctrl",layer="log1p")

/tmp/ipykernel_1563879/733474065.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  by_pert = [pert for _, pert in norm_data.obs.groupby(norm_data.obs["perturbation"])]
/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:458: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.stats[group_name, "names"] = self.var_names[global_indices]
/storage/mi/sommereg03/vbsc_2/lib/python3.13/site-packages/scanpy/tools/_rank_genes_groups.py:460: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

# Visual comparison:

In [ ]:
# visualization method: heatmap

# Shared DEGs:

## Get genes:

In [5]:
# Returns a dict of dicts: each dataset has a dict with perturbations as keys and the DEGs for these perturbations as values
def get_degs(adata_dict):
    degs = {}
    for name in names:
        degs[name] = {}
        for pert in tqdm(perts[name]):
            degs[name][pert] = adata_dict[name].uns["rank_genes_groups"]["names"][pert][adata_dict[name].uns["rank_genes_groups"]["pvals_adj"][pert] < 0.05]
    return degs

In [ ]:
# Get genes whose mean expression value in a certain perturbation is higher than the mean of means
def get_on_genes(adata_dict):
    on_genes = {}
    for name in names:
        on_genes[name] = {}
        for pert in tqdm(perts[name]):
            on_genes[name][pert] = adata_dict[name].var_names[pert_means[name][pert]>np.mean(pert_means[name][pert])]
    return on_genes

In [7]:
# For negative control: get random number of genes equal to no. of genes per pert in pGRiNS
def get_rand_dict(gene_dict):
    dict_rand = {}
    for name in names:
        dict_rand[name] = {}
        for pert in tqdm(perts[name]):
            dict_rand[name][pert] = choice(list(pgrins[name].var_names),size=len(gene_dict[name][pert]))
    return dict_rand

In [8]:
exp_dict = {} # Has structure test -> name -> pert
model_dict = {} # Has structure test -> model -> name -> pert

exp_dict[tests[0]] = get_degs(exp)
#exp_dict[tests[1]] = get_on_genes(exp)
"""
model_dict[tests[0]] = {}
model_dict[tests[1]] = {}
model_dict[tests[0]][models[0]] = get_degs(pgrins)
model_dict[tests[1]][models[0]] = get_on_genes(pgrins)
model_dict[tests[0]][models[1]] = get_rand_dict(model_dict[tests[0]][models[0]])
model_dict[tests[1]][models[1]] = get_rand_dict(model_dict[tests[1]][models[0]])
"""

100%|██████████| 963/963 [00:00<00:00, 9249.13it/s]


'\nmodel_dict[tests[0]] = {}\nmodel_dict[tests[1]] = {}\nmodel_dict[tests[0]][models[0]] = get_degs(pgrins)\nmodel_dict[tests[1]][models[0]] = get_on_genes(pgrins)\nmodel_dict[tests[0]][models[1]] = get_rand_dict(model_dict[tests[0]][models[0]])\nmodel_dict[tests[1]][models[1]] = get_rand_dict(model_dict[tests[1]][models[0]])\n'

## Chi squared:

In [ ]:
chi2_res_all = {} # Has structure test -> name -> model (list of pert_score)
for test in tests:
    chi2_res_all[test] = {}
    for name in names:
        chi2_res_all[test][name] = {}
        for model in models:
            chi2_res_all[test][name][model] = []
            for pert in perts[name]:
                degs_exp_set = set(exp_dict[test][name][pert])
                degs_model_set = set(model_dict[model][name][pert])
                all_genes = set(exp[name].var_names)

                cont_matrix = np.array([[len(degs_exp_set & degs_pgrins_set),len(degs_pgrins_set - degs_exp_set)],[len(degs_exp_set - degs_pgrins_set),len(all_genes - (degs_exp_set|degs_pgrins_set))]])
                res = chi2_contingency(cont_matrix)
                chi2_res_all[test][name][model].append(res.pvalue)

In [ ]:
# plot chi squared pval hist for perts
# for comparison: randomly assign genes as DEGs and calculate chi squared stat between that and exp data
# then do t test (or wilcoxon or sth) between distributions

In [ ]:
# How many genes are ON/OFF in both?

# Compare using metrics:

## Interpolated mean:

In [ ]:
# plot hist of distance between interpolated µ and each pert mean

In [ ]:
# or: across perts: plot mean MSE between adata_mean and adata_cell, and adata_mean and pgrins_cell

## Weighted metrics:

In [ ]:
# use WMSE, WR2, etc. to compare µ_c,exp and each µ_p,syn to GT of pert

# For weights calculation of interpolated duplicate etc. in GRiNS: is the order of pval_adj or t score alphabetically? Or are they reordered according to pval?

## GSEA:

- Inspired from https://www.sc-best-practices.org/conditions/gsea_pathway.html

In [ ]:
# Retrieving via python
msigdb = dc.op.resource("MSigDB")

# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))].rename(columns={"genesymbol":"target","geneset":"source"})
"""
# Retrieving via python
msigdb = dc.get_resource("MSigDB")

# Get reactome pathways
reactome = msigdb.query("collection == 'reactome_pathways'")
# Filter duplicates
reactome = reactome[~reactome.duplicated(("geneset", "genesymbol"))]#.rename(columns={"genesymbol":"target","geneset":"source"})
"""

In [36]:
gsea_dict = {}
for name in names:
    gsea_dict[name] = {}
    # Filter reactome by genes in each dataset
    gsea_dict[name]["reactome"] = reactome[reactome["target"].isin(exp[name].var_names)]
    # Get the genesets with no. of genes in [15,500]
    geneset_size = gsea_dict[name]["reactome"].groupby("source").size()
    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets

    for pert in tqdm(perts[name]):
        t_stats = sc.get.rank_genes_groups_df(exp[name], pert).set_index("names").sort_values("scores", key=np.abs, ascending=False)[["scores"]].rename_axis([pert], axis=1)
        scores, pvals = dc.mt.gsea(
            t_stats.T,
            gsea_dict[name]["reactome"][gsea_dict[name]["reactome"]["source"].isin(gsea_dict[name]["genesets"])],
        )
        gsea_dict[name][pert] = (
            pd.concat({"score": scores.T, "pval": pvals.T}, axis=1)
            .droplevel(level=1, axis=1)
            .sort_values("pval")
        )
        
"""
gsea_dict = {}
for name in names:
    gsea_dict[name] = {}
    # Filter reactome by genes in each dataset
    gsea_dict[name]["reactome"] = reactome[reactome["genesymbol"].isin(exp[name].var_names)]
    # Get the genesets with no. of genes in [15,500]
    geneset_size = gsea_dict[name]["reactome"].groupby("geneset").size()
    gsea_dict[name]["genesets"] = geneset_size.index[(geneset_size > 15) & (geneset_size < 500)] # ~600 genesets for both datasets
"""

  0%|          | 0/236 [00:00<?, ?it/s]

  0%|          | 3/963 [00:13<1:13:17,  4.58s/it]


SystemError: CPUDispatcher(<function _stsgsea at 0x7f4078593380>) returned a result with an exception set

In [ ]:
# get rank of pathways
# plot median distance between ranks for GRiNS and ranks for pert

## Coexpression Graph:
- Inspired from https://link.springer.com/protocol/10.1007/978-1-0716-2067-0_19
- https://cox-labs.github.io/coxdocs/WGCNA.html

In [15]:
%load_ext rpy2.ipython

In [20]:
%%R
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c("textshaping","Hmisc","WGCNA", "devtools"))

* installing *source* package ‘fs’ ...
** this is package ‘fs’ version ‘2.1.0’
** Paket ‘fs’ erfolgreich entpackt und MD5 Summen überprüft
** using staged installation
Package libuv was not found in the pkg-config search path.
Perhaps you should add the directory containing `libuv.pc'
to the PKG_CONFIG_PATH environment variable
Package 'libuv', required by 'virtual:world', not found
ERROR: configuration failed for package ‘fs’
* removing ‘/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/fs’


Using PKG_CFLAGS=
Using PKG_LIBS=-luv
--------------------------- [CONFIGURE] --------------------------------
Configuration failed because libuv was not found. Try installing:
 * deb: libuv1-dev (Debian, Ubuntu, etc)
 * rpm: libuv-devel (Fedora, EPEL)
 * brew: libuv (OSX)
Alternatively set environment variable USE_BUNDLED_LIBUV=1 to build a static
version of libuv that is included with this package.
-------------------------- [ERROR MESSAGE] ---------------------------
<stdin>:1:10: fatal error: uv.h: Datei oder Verzeichnis nicht gefunden
compilation terminated.
--------------------------------------------------------------------


* installing *source* package ‘textshaping’ ...
** this is package ‘textshaping’ version ‘1.0.5’
** Paket ‘textshaping’ erfolgreich entpackt und MD5 Summen überprüft
** using staged installation
Package fribidi was not found in the pkg-config search path.
Perhaps you should add the directory containing `fribidi.pc'
to the PKG_CONFIG_PATH environment variable
Package 'fribidi', required by 'virtual:world', not found


Using PKG_CFLAGS=-I/usr/include/harfbuzz -I/usr/include/freetype2 -I/usr/include/libpng16 -I/usr/include/glib-2.0 -I/usr/include/fribidi
Using PKG_LIBS=-lfreetype -lharfbuzz -lfribidi -lpng


** libs
using C++ compiler: ‘g++ (Debian 14.2.0-19) 14.2.0’


rm -f textshaping.so cpp11.o face_feature.o hb_shaper.o init.o string_bidi.o string_metrics.o string_shape.o
g++ -std=gnu++17 -I"/usr/share/R/include" -DNDEBUG -DNDEBUG -I/usr/include/harfbuzz -I/usr/include/freetype2 -I/usr/include/libpng16 -I/usr/include/glib-2.0 -I/usr/include/fribidi -I'/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/cpp11/include' -I'/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/systemfonts/include'     -fpic  -g -O2 -ffile-prefix-map=/home/jranke/git/r-backports/trixie/r-base-4.5.3=. -fstack-protector-strong -fstack-clash-protection -Wformat -Werror=format-security -fcf-protection -Wdate-time -D_FORTIFY_SOURCE=2   -c cpp11.cpp -o cpp11.o
g++ -std=gnu++17 -I"/usr/share/R/include" -DNDEBUG -DNDEBUG -I/usr/include/harfbuzz -I/usr/include/freetype2 -I/usr/include/libpng16 -I/usr/include/glib-2.0 -I/usr/include/fribidi -I'/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/cpp11/include' -I'/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/systemfo

string_bidi.cpp:12:10: fatal error: fribidi.h: Datei oder Verzeichnis nicht gefunden
   12 | #include <fribidi.h>
      |          ^~~~~~~~~~~
compilation terminated.
make: *** [/usr/lib/R/etc/Makeconf:211: string_bidi.o] Fehler 1
ERROR: compilation failed for package ‘textshaping’
* removing ‘/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/textshaping’
ERROR: dependency ‘fs’ is not available for package ‘sass’
Perhaps try a variation of:
install.packages('fs')
* removing ‘/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/sass’
ERROR: dependency ‘textshaping’ is not available for package ‘ragg’
Perhaps try a variation of:
install.packages('textshaping')
* removing ‘/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/ragg’
ERROR: dependency ‘fs’ is not available for package ‘usethis’
Perhaps try a variation of:
install.packages('fs')
* removing ‘/home/mi/sommereg03/R/x86_64-pc-linux-gnu-library/4.5/usethis’
ERROR: dependency ‘fs’ is not available for package ‘pkgload’
Perhap

'getOption("repos")' replaces Bioconductor standard repositories, see
'help("repositories", package = "BiocManager")' for details.
Replacement repositories:
    CRAN: https://cloud.r-project.org
Bioconductor version 3.22 (BiocManager 1.30.27), R 4.5.3 (2026-03-11)
Installing package(s) 'textshaping', 'Hmisc', 'WGCNA', 'devtools'
installiere auch Abhängigkeiten ‘sass’, ‘htmlwidgets’, ‘bslib’, ‘shiny’, ‘ragg’, ‘htmlTable’, ‘rmarkdown’, ‘usethis’, ‘fs’, ‘miniUI’, ‘pkgdown’, ‘pkgload’, ‘profvis’, ‘roxygen2’, ‘testthat’

versuche URL 'https://cloud.r-project.org/src/contrib/sass_0.4.10.tar.gz'
versuche URL 'https://cloud.r-project.org/src/contrib/htmlwidgets_1.6.4.tar.gz'
versuche URL 'https://cloud.r-project.org/src/contrib/bslib_0.11.0.tar.gz'
versuche URL 'https://cloud.r-project.org/src/contrib/shiny_1.13.0.tar.gz'
versuche URL 'https://cloud.r-project.org/src/contrib/ragg_1.5.2.tar.gz'
versuche URL 'https://cloud.r-project.org/src/contrib/htmlTable_2.5.0.tar.gz'
versuche URL 'https://c

In [ ]:
# Idea: use it to find modules and check them for pathways
# Other idea: look at Genetik Übung again and follow that

In [12]:
cor = pd.DataFrame(exp_matrix.T,columns=exp[name].var_names).corr()

In [13]:
cor

,AAK1,ABCA7,ABCB10,ABCB8,ABCC1,ABCC4,ABCC5,ABCE1,ABHD11,ABHD12,...,ZNF787,ZNF827,ZNF83,ZNF92,ZNFX1,ZNHIT2,ZNRF1,ZRANB2,ZYX,ZZZ3
AAK1,1.000000,0.310464,-0.436505,-0.411370,0.072246,-0.013140,-0.336226,-0.656188,-0.041883,0.130898,...,-0.577822,0.474967,0.579772,-0.095492,-0.186726,-0.312283,0.162075,-0.371478,0.266424,-0.133486
ABCA7,0.310464,1.000000,0.094064,0.044469,0.109071,0.082418,-0.089920,-0.246034,-0.054625,0.068941,...,-0.240950,0.204999,0.237966,-0.068759,-0.054733,-0.114779,0.085632,-0.287054,-0.252706,0.055463
ABCB10,-0.436505,0.094064,1.000000,0.329080,0.077646,0.095245,0.318359,0.257226,-0.115331,-0.115409,...,0.075804,-0.165982,-0.082217,0.018995,0.116544,-0.015134,-0.096504,0.022197,-0.511144,0.160705
ABCB8,-0.411370,0.044469,0.329080,1.000000,0.025414,0.215145,0.380099,0.467880,0.008613,-0.045580,...,0.275599,-0.158065,-0.343879,0.126664,0.238761,0.262034,-0.226160,0.308802,-0.183120,0.150453
ABCC1,0.072246,0.109071,0.077646,0.025414,1.000000,0.100381,-0.116518,-0.099041,-0.044468,0.046808,...,-0.156551,0.214580,0.163826,-0.137925,0.125933,0.026201,0.038598,-0.028526,-0.152800,-0.023397
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNHIT2,-0.312283,-0.114779,-0.015134,0.262034,0.026201,-0.234230,0.166399,0.422487,0.165938,0.083944,...,0.445578,-0.198975,-0.347898,-0.025279,0.164241,1.000000,-0.022979,0.293283,0.132228,0.081639
ZNRF1,0.162075,0.085632,-0.096504,-0.226160,0.038598,-0.127236,-0.259189,-0.142096,0.080312,-0.148435,...,-0.038207,0.039836,0.106627,-0.009994,-0.258432,-0.022979,1.000000,-0.192131,0.048305,-0.066859
ZRANB2,-0.371478,-0.287054,0.022197,0.308802,-0.028526,0.100962,0.360705,0.616338,0.035593,-0.057639,...,0.462719,-0.083502,-0.338972,0.109228,0.213865,0.293283,-0.192131,1.000000,0.163902,0.170307
ZYX,0.266424,-0.252706,-0.511144,-0.183120,-0.152800,-0.260293,-0.143427,-0.048488,0.079927,0.029584,...,0.130338,-0.009178,-0.023581,0.062912,-0.076762,0.132228,0.048305,0.163902,1.000000,-0.020921


In [ ]:
# Compare networks for exp & pGRiNS with CoDiNA: file:///home/gesomme/Documents/Uni/B_Semester_5/Genetik/%C3%9Cbung/Tutorial%20on%20Gene%20Expression%20and%20Network%20Analysis.html
# Look at modules and perform GO enrichment analysis or sth

# GSEA is done perturbation wise (can pGRiNS capture the specific pathways affected by a perturbation?)
# GO is done across perturbations (are genes generally coexpressed in such a way that networks/pathways are derivable?   )

## GO Enrichment: